# Exercise 4.1: Data/MC around the Z peak

From *Programming in High Energy Particle Physics*, Chapter 4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch04/ex04_1_solution.ipynb)

Exercise 4.1 Data/MC around the Z peak Create a data/MC comparison plot for the dilepton mass spectrum around the Z boson peak. Include proper error bars, a ratio panel, and publication-quality labels.

<details><summary>Hint</summary>

Adapt Listing 4.1 or Listing 4.6 : use a Breit–Wigner (or Gaussian) Z component and a falling exponential background as the “MC”, and generate pseudo-data by Poisson-fluctuating their sum bin by bin. The events.root ntuple from Listing 3.6 can supply the Z component.

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q numpy matplotlib")

### Build and plot toy data/MC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng=np.random.default_rng(42)
edges=np.linspace(60,120,31);centers=0.5*(edges[1:]+edges[:-1])
widths=np.diff(edges)
gamma=2.4955; m_z=91.1876
z_shape=1/((centers-m_z)**2+(gamma/2)**2)
z_mc=1800*z_shape/z_shape.sum()
bg_shape=np.exp(-(centers-60)/25)
bg_mc=260*bg_shape/bg_shape.sum()
mc=z_mc+bg_mc
data=rng.poisson(mc)
ratio=np.divide(data,mc,out=np.full_like(mc,np.nan),where=mc>0)
ratio_err=np.divide(np.sqrt(data),mc,out=np.zeros_like(mc),where=mc>0)
fig,(ax,rax)=plt.subplots(2,1,figsize=(8,7),sharex=True,
    gridspec_kw={'height_ratios':[3,1],'hspace':0.05})
ax.bar(centers,bg_mc,width=widths,color='#0072B2',label='Background',align='center')
ax.bar(centers,z_mc,bottom=bg_mc,width=widths,color='#E69F00',label='Z → ℓℓ',align='center')
ax.errorbar(centers,data,yerr=np.sqrt(data),fmt='ko',markersize=3,label='Pseudo-data')
ax.set_ylabel('Events / 2 GeV');ax.legend(frameon=False)
ax.text(0.03,0.96,'CMS-style  Toy simulation',transform=ax.transAxes,va='top',fontweight='bold')
rax.errorbar(centers,ratio,yerr=ratio_err,fmt='ko',markersize=3)
rax.axhline(1,color='black',lw=1);rax.set_ylim(0.5,1.5)
rax.set_xlabel(r'$m_{\ell\ell}$ [GeV]');rax.set_ylabel('Data/MC')
fig.savefig('datamc_z.png',dpi=150,bbox_inches='tight');plt.show()
print('Total MC and pseudo-data:',mc.sum(),data.sum())

In [ ]:
assert np.isfinite(ratio).all() and 0.8 < data.sum()/mc.sum() < 1.2

### Interpretation

The ratio fluctuates around one because pseudo-data are drawn from the displayed MC expectation. These are synthetic events, not measured CMS data.